In [6]:
import os
from glob import glob
import xml.etree.ElementTree as ET
from ultralytics import YOLO
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Compose, Resize, ToTensor
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import shutil

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using Device: {device}')

Using Device: mps


In [ ]:
base_path = "/Users/aum/Developer/DL-Project/backend/IDD_Detection"
images_path = os.path.join(base_path, "JPEGImages")
annotations_path = os.path.join(base_path, "Annotations")
output_path = "/Users/aum/Developer/DL-Project/backend/idd_yolo"
os.makedirs(output_path, exist_ok=True)

classes = [
    "car", "person", "autorickshaw", "truck", "bus", "motorcycle",
    "bicycle", "animal", "traffic_light", "traffic_sign",
    "vehicle fallback", "animal fallback", "unlabeled"
]
class_to_idx = {cls: i for i, cls in enumerate(classes)}

# Create folders
for split in ["train", "val", "test"]:
    os.makedirs(f"{output_path}/images/{split}", exist_ok=True)
    os.makedirs(f"{output_path}/labels/{split}", exist_ok=True)

def convert_split(split_name):
    txt_file = os.path.join(base_path, f"{split_name}.txt")
    with open(txt_file, "r") as f:
        image_ids = [line.strip() for line in f.readlines()]
    for img_id in tqdm(image_ids):
        img_file = os.path.join(images_path, img_id + ".jpg")
        xml_file = os.path.join(annotations_path, img_id + ".xml")
        if not os.path.exists(xml_file) or not os.path.exists(img_file):
            continue
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)
        yolo_lines = []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in class_to_idx: continue
            class_id = class_to_idx[label]
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            x_center = ((xmin + xmax)/2)/w
            y_center = ((ymin + ymax)/2)/h
            box_width = (xmax - xmin)/w
            box_height = (ymax - ymin)/h
            yolo_lines.append(f"{class_id} {x_center} {y_center} {box_width} {box_height}")
        if len(yolo_lines)==0: continue
        shutil.copy(img_file, os.path.join(output_path, "images", split_name, img_id + ".jpg"))
        with open(os.path.join(output_path, "labels", split_name, img_id + ".txt"), "w") as f:
            f.write("\n".join(yolo_lines))

# Convert all splits
convert_split("train")
convert_split("val")
convert_split("test")
print("✅ Conversion Complete!")

100%|███████████████████████████████████| 4794/4794 [00:00<00:00, 697329.40it/s]

✅ Conversion Complete!


In [ ]:
yaml_content = f"""
path: {output_path}
train: images/train
val: images/val
test: images/test

names:
  0: car
  1: person
  2: autorickshaw
  3: truck
  4: bus
  5: motorcycle
  6: bicycle
  7: animal
  8: traffic_light
  9: traffic_sign
  10: vehicle_fallback
  11: animal_fallback
  12: unlabeled
"""
yaml_path = os.path.join("/Users/aum/Developer/backend/DL-Project", "idd.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print("✅ idd.yaml created")

✅ idd.yaml created


In [10]:
model = YOLO("yolov8s.pt") 

model.train(
    data=yaml_path,
    epochs=30,     
    imgsz=640,
    batch=4,
    device="mps",  
    patience=5,
    cache=True
)

Ultralytics 8.4.21 🚀 Python-3.13.9 torch-2.10.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/aum/Developer/DL-Project/idd.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, perspective=0.0, plot

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x14fecd940>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    

In [1]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

metrics = model.val(
    data="idd.yaml",
    device="mps"
)

# Print only accuracy metrics
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.21 🚀 Python-3.13.9 torch-2.10.0 MPS (Apple M4 Pro)
Model summary (fused): 73 layers, 11,130,615 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 1287.6±218.8 MB/s, size: 428.1 KB)
val: Scanning /Users/aum/Developer/DL-Project/idd_yolo/labels/val/frontFar/BLR-2018-04-16_16-14-27_frontFar.cache... 2839 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2839/2839 360.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 1.9it/s 1:320.3ss
                   all       2839      11871       0.72      0.509      0.561      0.377
                   car       1481       2521      0.751      0.719      0.756      0.547
                person       1032       2584      0.789      0.491      0.581      0.318
          autorickshaw        432        508      0.779      0.593      0.667      0.457
                 truck        857       1027      0.782      0.774      0.81

In [5]:
import matplotlib.pyplot as plt

# After evaluation
metrics = model.val(data="idd.yaml", device="mps")

# Example: plot mAP50 per class
class_names = metrics.box.names
mAP50 = metrics.box.map50_cls  # list of mAP50 per class

plt.figure(figsize=(10,5))
plt.bar(class_names, mAP50)
plt.xticks(rotation=45)
plt.ylabel("mAP50")
plt.title("Class-wise mAP50")
plt.show()

Ultralytics 8.4.21 🚀 Python-3.13.9 torch-2.10.0 MPS (Apple M4 Pro)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 972.3±213.7 MB/s, size: 503.2 KB)
val: Scanning /Users/aum/Developer/DL-Project/idd_yolo/labels/val/frontFar/BLR-2018-04-16_16-14-27_frontFar.cache... 2839 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2839/2839 1.3Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 3.5it/s 51.5s0.3ss
                   all       2839      11871       0.72      0.509      0.561      0.377
                   car       1481       2521      0.751      0.719      0.756      0.547
                person       1032       2584      0.789      0.491      0.581      0.318
          autorickshaw        432        508      0.779      0.593      0.667      0.457
                 truck        857       1027      0.782      0.774      0.819      0.655
                   bus        827       1181      0.867      0.767     

AttributeError: 'Metric' object has no attribute 'names'. See valid attributes below.
Class for computing evaluation metrics for Ultralytics YOLO models.

Attributes:
    p (list): Precision for each class. Shape: (nc,).
    r (list): Recall for each class. Shape: (nc,).
    f1 (list): F1 score for each class. Shape: (nc,).
    all_ap (list): AP scores for all classes and all IoU thresholds. Shape: (nc, 10).
    ap_class_index (list): Index of class for each AP score. Shape: (nc,).
    nc (int): Number of classes.

Methods:
    ap50: AP at IoU threshold of 0.5 for all classes.
    ap: AP at IoU thresholds from 0.5 to 0.95 for all classes.
    mp: Mean precision of all classes.
    mr: Mean recall of all classes.
    map50: Mean AP at IoU threshold of 0.5 for all classes.
    map75: Mean AP at IoU threshold of 0.75 for all classes.
    map: Mean AP at IoU thresholds from 0.5 to 0.95 for all classes.
    mean_results: Mean of results, returns mp, mr, map50, map.
    class_result: Class-a

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Path to YOLOv8 training results
results_csv = "runs/detect/train/results.csv"

# Load results
results = pd.read_csv(results_csv)

# Plot Losses
plt.figure(figsize=(12,5))
plt.plot(results['epoch'], results['box_loss'], label='Box Loss')
plt.plot(results['epoch'], results['cls_loss'], label='Class Loss')
plt.plot(results['epoch'], results['obj_loss'], label='Object Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("YOLOv8 Training Losses")
plt.legend()
plt.show()

# Plot mAP
plt.figure(figsize=(12,5))
plt.plot(results['epoch'], results['mAP_0.5'], label='mAP50')
plt.plot(results['epoch'], results['mAP_0.5:0.95'], label='mAP50-95')
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("YOLOv8 mAP Metrics")
plt.legend()
plt.show()

KeyError: 'box_loss'

In [8]:
from ultralytics import YOLO

# Load best model
model = YOLO("runs/detect/train/weights/best.pt")

# Evaluate on validation set
metrics = model.val(
    data="idd.yaml",
    device="mps"
)

# Print overall metrics
print(f"mAP50: {metrics.map50:.3f}")
print(f"mAP50-95: {metrics.map:.3f}")
print(f"Precision: {metrics.mp:.3f}")
print(f"Recall: {metrics.mr:.3f}")

# Class names from the model
class_names = model.names  # dictionary {0: 'car', 1: 'person', ...}

# Class-wise metrics
print("\nClass-wise AP50 and Recall:")
for i in range(metrics.nc):
    ap50 = metrics.class_result(i).ap50   # AP50 for class i
    recall = metrics.class_result(i).r     # Recall for class i
    print(f"{class_names[i]:<20} | AP50: {ap50:.3f} | Recall: {recall:.3f}")

Ultralytics 8.4.21 🚀 Python-3.13.9 torch-2.10.0 MPS (Apple M4 Pro)
Model summary (fused): 73 layers, 11,130,615 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4768.4±3010.0 MB/s, size: 537.8 KB)
val: Scanning /Users/aum/Developer/DL-Project/idd_yolo/labels/val/frontFar/BLR-2018-04-16_16-14-27_frontFar.cache... 2839 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2839/2839 1.3Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 3.5it/s 51.1s0.3ss
                   all       2839      11871       0.72      0.509      0.561      0.377
                   car       1481       2521      0.751      0.719      0.756      0.547
                person       1032       2584      0.789      0.491      0.581      0.318
          autorickshaw        432        508      0.779      0.593      0.667      0.457
                 truck        857       1027      0.782      0.774      0.81

AttributeError: 'DetMetrics' object has no attribute 'map50'. See valid attributes below.
Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP).

Attributes:
    names (dict[int, str]): A dictionary of class names.
    box (Metric): An instance of the Metric class for storing detection results.
    speed (dict[str, float]): A dictionary for storing execution times of different parts of the detection process.
    task (str): The task type, set to 'detect'.
    stats (dict[str, list]): A dictionary containing lists for true positives, confidence scores, predicted classes,
        target classes, and target images.
    nt_per_class: Number of targets per class.
    nt_per_image: Number of targets per image.

Methods:
    update_stats: Update statistics by appending new values to existing stat collections.
    process: Process predicted results for object detection and update metrics.
    clear_stats: Clear the stored statistics.
    keys